In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [4]:
import pandas as pd, time
from tqdm import tqdm
from woc.remote import WocMapsRemote

sel = pd.DataFrame([
    ('lsstdesc_weaklensingdeblending',  'https://github.com/LSSTDESC/WeakLensingDeblending'),
    ('mkazhdan_poissonrecon',           'https://github.com/mkazhdan/PoissonRecon'),
    ('julianlsolvers_nlsolversbase.jl', 'https://github.com/JuliaNLSolvers/NLSolversBase.jl'),
    ('gridtools_gridtools',             'https://github.com/GridTools/gridtools'),
    ('christophsax_seasonal',           'https://github.com/christophsax/seasonal'),
    ('svf-tools_svf',                   'https://github.com/SVF-tools/SVF'),
    ('stephenslab_flashr',              'https://github.com/stephenslab/flashr'),
    ('feihoo87_qulab',                  'https://github.com/feihoo87/QuLab'),
    ('soerenpannier_emdi',              'https://github.com/SoerenPannier/emdi'),
    ('gimli-org_gimli',                 'https://github.com/gimli-org/gimli'),
], columns=['wocid', 'url'])

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

list_df = []
for row in sel.itertuples():
    try:
        shas = woc.get_values('p2c', row.wocid)
    except Exception as e:
        print('FAILED', row.wocid, e); continue
    print(row.wocid, len(shas))
    d = pd.DataFrame(shas, columns=['sha1']); d['project'] = row.wocid
    list_df.append(d)
    time.sleep(1)

df = pd.concat(list_df, ignore_index=True)
df.to_csv('df_commits.csv', index=False)

lsstdesc_weaklensingdeblending 735
mkazhdan_poissonrecon 414
julianlsolvers_nlsolversbase.jl 480
gridtools_gridtools 16440
christophsax_seasonal 1227
svf-tools_svf 6530
stephenslab_flashr 616
feihoo87_qulab 4361
soerenpannier_emdi 1143
gimli-org_gimli 7466


In [13]:
import os
os.makedirs('chunks', exist_ok=True)

for prj, g in df.groupby('project'):
    fn = f'chunks/{prj}.csv'
    if os.path.exists(fn):
        print('skipping (done):', prj); continue
    shas = g['sha1'].tolist()
    rows = []
    for i in tqdm(range(0, len(shas), 10), desc=prj):
        chunk = shas[i:i+10]
        for attempt in range(3):                      # retry up to 3 times
            try:
                res, err = woc.get_values_many('commit.tch', chunk)
                if err: print('Got Errors', err)
                res = {k: v[0] for k, v in res.items()}
                break
            except Exception as e:
                print('retry', attempt, e); time.sleep(5)
        else:
            print('GAVE UP on chunk in', prj); continue
        for sha, c in res.items():
            rows.append({'project_wocid': prj, 'commit_sha1': sha,
                         'author': c[2][0], 'time': int(c[2][1]),
                         'commit message': c[4]})
        time.sleep(1)
    print(prj, 'expected', len(shas), 'got', len(rows))
    pd.DataFrame(rows).to_csv(fn, index=False)

christophsax_seasonal: 100%|█████████████████████████████████████████████████████████| 123/123 [02:06<00:00,  1.03s/it]


christophsax_seasonal expected 1227 got 1227


feihoo87_qulab:  30%|███████████████████▏                                            | 131/437 [02:15<05:17,  1.04s/it]

Got Errors {'4df6f45fb773ecc76f1d59b1ab0cf6306911f296': 'Key 4df6f45fb773ecc76f1d59b1ab0cf6306911f296 not found in /da5_fast/All.sha1c/commit_77.tch'}


feihoo87_qulab: 100%|████████████████████████████████████████████████████████████████| 437/437 [07:34<00:00,  1.04s/it]


...

Got Errors {'5bf807179867e08e30eb9760842fd094d822c2f1': 'Key 5bf807179867e08e30eb9760842fd094d822c2f1 not found in /da5_fast/All.sha1c/commit_91.tch'}


svf-tools_svf:  83%|█████████████████████████████████████████████████████▊           | 541/653 [09:24<01:56,  1.04s/it]

Got Errors {'d5d8128d8d54cb037ad17be6afd4b8783c32e258': 'Key d5d8128d8d54cb037ad17be6afd4b8783c32e258 not found in /da5_fast/All.sha1c/commit_85.tch'}


svf-tools_svf: 100%|█████████████████████████████████████████████████████████████████| 653/653 [11:21<00:00,  1.04s/it]

svf-tools_svf expected 6530 got 6527


In [6]:
import os
out = pd.concat([pd.read_csv(f'chunks/{f}') for f in sorted(os.listdir('chunks'))],
                ignore_index=True)
out.to_csv('tcartier_project_summary.csv', index=False, sep=';')
print(len(out), 'rows')
out.head(3)

39388 rows


,project_wocid,commit_sha1,author,time,commit message
0,christophsax_seasonal,00352d722b0b33cc67cf506491555f1c455297bd,Christoph Sax <christoph.sax@gmail.com>,1600417136,Merge pull request #253 from christophsax/stat...
1,christophsax_seasonal,003d9dd00c851c838f2c688bb21116da19dfad62,Christoph <christoph.sax@gmail.com>,1385979532,update hidden extension remark\n
2,christophsax_seasonal,0053e34f6fcce45cb28508877d0c6f918b0fd7e9,Christoph Sax <christoph.sax@gmail.com>,1600414334,"rm_defaults, resolves #251\n"


In [7]:
chk = pd.read_csv('tcartier_project_summary.csv', sep=';')
print(chk.columns.tolist())   # ['project_wocid','commit_sha1','author','time','commit message']
print(len(chk), 'rows')       # should match len(out)
print(chk['project_wocid'].nunique(), 'projects')   # should be 10
print(chk.isna().sum())       # a few empty messages are fine

['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
39388 rows
10 projects
project_wocid      0
commit_sha1        0
author             0
time               0
commit message    31
dtype: int64


In [8]:
!ls -lh tcartier_project_summary.csv

-rw-------. 1 tcartier tcartier 6.2M Sep 29 14:41 tcartier_project_summary.csv


In [9]:
out['email'] = out['author'].str.extract(r'<(.+?)>')
out['dt'] = pd.to_datetime(out['time'], unit='s')
woc_summary = out.groupby('project_wocid').agg(
    commits=('commit_sha1', 'nunique'),
    authors_by_string=('author', 'nunique'),
    authors_by_email=('email', 'nunique'),
    min_time=('dt', 'min'),
    max_time=('dt', 'max'))
woc_summary

,commits,authors_by_string,authors_by_email,min_time,max_time
project_wocid,,,,,
christophsax_seasonal,1227,17,16,2013-09-17 20:14:42,2025-02-28 17:18:20
feihoo87_qulab,4360,11,10,2014-12-09 11:09:39,2025-09-08 12:17:40
gimli-org_gimli,7454,77,63,2011-05-06 17:56:03,2025-11-03 19:23:44
gridtools_gridtools,16438,106,90,2013-10-25 08:01:33,2026-03-03 10:45:08
julianlsolvers_nlsolversbase.jl,476,47,45,2016-01-26 10:00:10,2025-06-26 02:32:58
lsstdesc_weaklensingdeblending,735,26,22,2013-03-11 11:27:58,2026-01-16 19:46:14
mkazhdan_poissonrecon,414,58,48,2015-07-14 23:31:43,2025-10-25 01:00:38
soerenpannier_emdi,1143,23,19,2016-10-28 12:35:37,2025-09-05 12:35:30
stephenslab_flashr,614,7,5,2017-09-12 19:03:25,2024-12-04 13:11:19


In [11]:
import requests, pandas as pd

repos = {
 'lsstdesc_weaklensingdeblending':  'LSSTDESC/WeakLensingDeblending',
 'mkazhdan_poissonrecon':           'mkazhdan/PoissonRecon',
 'julianlsolvers_nlsolversbase.jl': 'JuliaNLSolvers/NLSolversBase.jl',
 'gridtools_gridtools':             'GridTools/gridtools',
 'christophsax_seasonal':           'christophsax/seasonal',
 'svf-tools_svf':                   'SVF-tools/SVF',
 'stephenslab_flashr':              'stephenslab/flashr',
 'feihoo87_qulab':                  'feihoo87/QuLab',
 'soerenpannier_emdi':              'SoerenPannier/emdi',
 'gimli-org_gimli':                 'gimli-org/gimli',
}
headers = {}   # or {"Authorization": "Bearer YOUR_TOKEN"} (don't commit the token)

gh = []
for wocid, o_r in repos.items():
    r = requests.get(f'https://api.github.com/repos/{o_r}', headers=headers).json()
    c = requests.get(f'https://api.github.com/repos/{o_r}/commits?per_page=1', headers=headers).json()
    gh.append({'project': wocid, 'stars': r['stargazers_count'],
               'forks': r['forks_count'], 'last_commit': c[0]['commit']['committer']['date']})
gh_df = pd.DataFrame(gh)
gh_df

,project,stars,forks,last_commit
0,lsstdesc_weaklensingdeblending,15,15,2025-07-24T20:47:53Z
1,mkazhdan_poissonrecon,1860,470,2026-04-29T15:37:55Z
2,julianlsolvers_nlsolversbase.jl,34,31,2026-08-17T16:50:53Z
3,gridtools_gridtools,68,24,2026-02-18T13:11:51Z
4,christophsax_seasonal,120,30,2026-09-17T18:04:00Z
5,svf-tools_svf,1711,498,2026-09-29T11:52:00Z
6,stephenslab_flashr,54,11,2024-12-04T13:11:19Z
7,feihoo87_qulab,18,20,2026-07-14T15:00:58Z
8,soerenpannier_emdi,17,24,2025-08-23T17:57:56Z
9,gimli-org_gimli,509,163,2026-09-24T10:31:28Z


### GitHub stats (retrieved Sept 29, 2026)
| project | stars | forks | last commit |
|---|---|---|---|
| lsstdesc_weaklensingdeblending | 15 | 15 | 2025-07-24 |
| mkazhdan_poissonrecon | 1860 | 470 | 2026-04-29 |
| julianlsolvers_nlsolversbase.jl | 34 | 31 | 2026-08-17 |
| gridtools_gridtools | 68 | 24 | 2026-02-18 |
| christophsax_seasonal | 120 | 30 | 2026-09-17 |
| svf-tools_svf | 1711 | 498 | 2026-09-29 |
| stephenslab_flashr | 54 | 11 | 2024-12-04 |
| feihoo87_qulab | 18 | 20 | 2026-07-14 |
| soerenpannier_emdi | 17 | 24 | 2025-08-23 |
| gimli-org_gimli | 509 | 163 | 2026-09-24 |

### WoC stats
| project | commits | authors (by email) | authors (by name string) | min time | max time |
|---|---|---|---|---|---|
| christophsax_seasonal | 1227 | 16 | 17 | 2013-09-17 | 2025-02-28 |
| feihoo87_qulab | 4360 | 10 | 11 | 2014-12-09 | 2025-09-08 |
| gimli-org_gimli | 7454 | 63 | 77 | 2011-05-06 | 2025-11-03 |
| gridtools_gridtools | 16438 | 90 | 106 | 2013-10-25 | 2026-03-03 |
| julianlsolvers_nlsolversbase.jl | 476 | 45 | 47 | 2016-01-26 | 2025-06-26 |
| lsstdesc_weaklensingdeblending | 735 | 22 | 26 | 2013-03-11 | 2026-01-16 |
| mkazhdan_poissonrecon | 414 | 48 | 58 | 2015-07-14 | 2025-10-25 |
| soerenpannier_emdi | 1143 | 19 | 23 | 2016-10-28 | 2025-09-05 |
| stephenslab_flashr | 614 | 5 | 7 | 2017-09-12 | 2024-12-04 |
| svf-tools_svf | 6527 | 196 | 238 | 2015-06-05 | 2025-11-06 |

Notes: 24 of 39,412 commits (0.06%) couldn't be retrieved after retries. Author counts by name string overcount because the same person shows up with name variants. WoC data extends beyond the Jan 2025 cutoff in the handout. For gridtools, weaklensingdeblending, and emdi, WoC's latest commit is later than GitHub's last commit, possibly from non-default branches/forks or author vs committer time.

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github